# 11 · Proyecto final: un asistente RAG supervisado

Diez módulos, diez piezas. Este proyecto final las ensambla en un único sistema que funciona sobre un corpus
**nuevo**, para que no puedas apoyarte en nada que hayas ajustado antes: la base de conocimiento de soporte de una
empresa ficticia de termostatos inteligentes, *Thermio*, más una pequeña tabla de ventas. Después recorre todo el
bucle del curso:

> construir → responder → trazar → **juzgar** → encontrar lo que falló → **arreglar la documentación** → reindexar → volver a medir

Todo es determinista y sin conexión: un generador de sustitución y un supervisor de sustitución ocupan el lugar de
los modelos de lenguaje, exactamente como en los módulos anteriores, de modo que los números que ves los produce
el código que puedes leer. Cambiar a modelos reales modifica la calidad de los dos sustitutos, no la arquitectura.

**Aprenderás**
1. Cómo encajan las piezas de los módulos 01 a 10 en un único pipeline, con un enrutador que manda cada pregunta a la recuperación de pasajes, a SQL o a un rechazo
2. A elegir el peso de recuperación y el umbral de rechazo con un **conjunto de desarrollo**, y a evaluar después con una batería distinta
3. Una respuesta tipada que no puede existir sin una cita (Pydantic)
4. Un supervisor que juzga cada respuesta, y si fiarse de él (medimos su acuerdo con las referencias)
5. Convertir los fallos en un arreglo de la documentación, reindexar y demostrar la mejora con intervalos de confianza
6. Una lista de comprobación antes de poner un sistema RAG delante de los usuarios

## 1. El sistema que vamos a construir

```
 pregunta ──► enrutador ──┬─ pasaje ─► recuperación (BM25 + embeddings, RRF) ─► generador ─┐
                          ├─ sql ────► text-to-SQL ─► guardián ─► DuckDB ──────────────────┤
                          └─ rechazo (no se encontró nada relevante) ──────────────────────┤
                                                                                            ▼
                     Answer tipada (citas obligatorias) ─► registro de trazas (tokens, coste, latencia)
                                                                  │
                                       el supervisor juzga cada respuesta ─► resumen diario
                                                                  │
                    análisis de fallos de recuperación ─► arreglo de la documentación ─► reindexar ─► volver a medir
```

| Componente | Construido en el módulo |
|---|---|
| troceado por estructura del documento, BM25, hit@k y MRR | 01 |
| embeddings y recuperación híbrida con Reciprocal Rank Fusion | 02 |
| salida tipada con Pydantic | 04 |
| un pipeline como pasos explícitos con un enrutador (LangGraph expresa lo mismo como un grafo) | 05 |
| respuestas desde datos con un guardián SQL de solo lectura | 06 |
| batería de preguntas, calificación, intervalos de confianza por bootstrap | 07 |
| el supervisor, el bucle que arregla la documentación | 08 |
| trazas, tokens, coste | 09 |
| rechazo, hábitos de seguridad, modelo de coste | 10 |

## 2. Los datos: una base de conocimiento, una tabla, tres conjuntos de preguntas

* **La base de conocimiento:** 25 pasajes de soporte, cada uno con un título de sección.
* **La tabla de ventas:** unidades e ingresos por mes, producto y región, cargada en DuckDB.
* **El conjunto de desarrollo:** 12 preguntas (8 paráfrasis y 4 que usan términos exactos como `E45`) usadas **solo** para elegir el peso de recuperación y el umbral de rechazo.
* **La batería:** 24 preguntas con respuesta conocida, usadas **solo** para evaluar: 16 respondibles desde pasajes,
  4 desde la tabla de ventas y 4 que nada puede responder (incluida una trampa con un código de error verosímil).

Mantener separados el desarrollo y la batería es la disciplina del módulo 07: ajusta con un conjunto, informa con el otro.

In [1]:
import re, math, json, random, time, itertools
from collections import defaultdict
from collections import Counter
import numpy as np
import duckdb
import matplotlib.pyplot as plt
from typing import Literal
from pydantic import BaseModel, model_validator, ValidationError
from sentence_transformers import SentenceTransformer

KB = [
    ("t01", "Setup / Wi-Fi", "The Thermio One connects to 2.4 GHz Wi-Fi only. Hold the ring button for 5 seconds until it blinks blue, then pick your network in the app."),
    ("t02", "Setup / Wiring", "Thermio One works with 24 V heating and cooling systems and needs a C-wire. If your system has no C-wire, install the included power adapter at the furnace."),
    ("t03", "Setup / Compatibility", "Thermio One is not compatible with 120 V or 240 V line-voltage baseboard heaters. Use Thermio Line for those systems."),
    ("t04", "Features / Schedules", "Schedules support up to 6 changes per day, with separate weekday and weekend programs. Changes apply within one minute of saving."),
    ("t05", "Features / Geofencing", "Away mode via geofencing starts when the last phone leaves a 500 m radius, and it sets the temperature back by 4 degrees."),
    ("t06", "Features / Eco mode", "Eco mode limits heating to 16 degrees and cooling to 28 degrees. The heating limit can be changed in settings between 10 and 18 degrees."),
    ("t07", "Troubleshooting / Offline", "If the thermostat shows Offline, restart the router, then hold the ring button for 10 seconds to reconnect. Settings are kept."),
    ("t08", "Troubleshooting / Blank display", "A blank display usually means no power. Check the C-wire and the furnace breaker. The battery backup keeps the display on for about 2 hours."),
    ("t09", "Troubleshooting / Error E21", "Error E21 means the temperature sensor reading is out of range. Remove the thermostat from its base for 30 seconds and refit it. If E21 returns, contact support for a replacement."),
    ("t10", "Troubleshooting / Error E45", "Error E45 means the system is short cycling. The thermostat waits 5 minutes between heating starts to protect the compressor."),
    ("t11", "Troubleshooting / Wrong temperature", "If the temperature reading seems off, avoid direct sun and drafts. A calibration offset of up to 3 degrees can be set in the app."),
    ("t12", "Warranty / Coverage", "Thermio One has a 3 year limited warranty from the date of purchase. Damage from water or from incorrect wiring is not covered."),
    ("t13", "Warranty / Returns", "Returns are accepted within 45 days of delivery if the device is undamaged. The refund is issued to the original payment method within 10 business days."),
    ("t14", "Warranty / Replacement", "A faulty unit under warranty is replaced with a refurbished or new device and shipped within 3 business days of approval."),
    ("t15", "Account / Password reset", "To reset the password, use the Forgot password link on the sign-in page. The reset email arrives within 5 minutes and the link is valid for 1 hour."),
    ("t16", "Account / Sharing", "Up to 8 people can share one home in the app. The owner can remove members at any time under Home settings."),
    ("t17", "Account / Data", "Temperature history is stored for 12 months. Deleting an account removes all history within 30 days."),
    ("t18", "Energy / Savings report", "The monthly energy report compares runtime with the previous month and with similar homes. Reports need at least 14 days of data."),
    ("t19", "Energy / Smart utility", "Utility programs can request a temporary setback of up to 2 degrees for at most 4 hours. You can opt out of any event from the app."),
    ("t20", "Hardware / Battery", "Thermio One uses a built-in rechargeable battery that charges from the C-wire. The battery is not user-replaceable."),
    ("t21", "Hardware / Sensors", "A remote room sensor (sold separately) measures temperature and occupancy up to 10 m from the thermostat. Up to 6 sensors can be paired."),
    ("t22", "Hardware / Dimensions", "The thermostat measures 86 mm in diameter and 24 mm in depth and weighs 150 g."),
    ("t23", "Support / Contact", "Support is available by chat from 08:00 to 20:00 on weekdays. Response time for email is within 24 hours."),
    ("t24", "Support / Firmware", "Firmware updates install automatically between 02:00 and 04:00 local time. An update takes about 10 minutes and the schedule keeps running."),
    ("t25", "Support / Privacy", "Thermio does not record audio or video. Location is used only for geofencing and can be switched off in the app."),
]

# sales table: month x product x region (deterministic synthetic numbers)
_r = random.Random(21)
PRICE = {"Thermio One": 129, "Thermio Line": 99, "Thermio Sensor": 39}
sales = []
for m in range(1, 7):
    for p in PRICE:
        for reg in ("North", "South"):
            u = _r.randint(40, 400)
            sales.append(dict(month=f"2026-{m:02d}", product=p, region=reg, units=u, revenue_eur=u * PRICE[p]))
con = duckdb.connect()
con.execute("CREATE TABLE sales (month VARCHAR, product VARCHAR, region VARCHAR, units INTEGER, revenue_eur INTEGER)")
con.executemany("INSERT INTO sales VALUES (?, ?, ?, ?, ?)", [tuple(r.values()) for r in sales])
print(len(KB), "pasajes;", len(sales), "filas de ventas;", sum(r["units"] for r in sales), "unidades en total")
print(sales[0])

/home/patito/Developments/academy/rag-systems/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


25 pasajes; 36 filas de ventas; 8245 unidades en total
{'month': '2026-01', 'product': 'Thermio One', 'region': 'North', 'units': 124, 'revenue_eur': 15996}


In [2]:
# references for the SQL questions come from an INDEPENDENT plain-Python calculation, never from the system under test
def tot(key, **where):
    return sum(r[key] for r in sales if all(r[k] == v for k, v in where.items()))
exp = dict(
    s01=str(tot("units", product="Thermio One")),
    s02=max(("North", "South"), key=lambda reg: tot("units", product="Thermio Line", month="2026-03", region=reg)),
    s03=str(tot("revenue_eur", month="2026-06")),
    s04=str(tot("units", product="Thermio Sensor", region="North")),
)
print("referencias de las preguntas SQL:", exp)

# (id, question, gold passage or None, the fact the answer must contain)
BATTERY = [
    ("c01", "Can I connect the thermostat to a 5 GHz network?", "t01", "2.4 GHz"),
    ("c02", "My heating system has no common wire, what now?", "t02", "power adapter"),
    ("c03", "Does it work with electric baseboard heaters?", "t03", "Thermio Line"),
    ("c04", "How many schedule changes can I set per day?", "t04", "6 changes"),
    ("c05", "When does the house switch to away mode?", "t05", "500 m"),
    ("c06", "What is the lowest heating limit in eco mode?", "t06", "16 degrees"),
    ("c07", "The thermostat says Offline, what should I do?", "t07", "restart the router"),
    ("c08", "How long does the display stay on when the power is cut?", "t08", "2 hours"),
    ("c09", "What does error E21 mean?", "t09", "sensor"),
    ("c10", "Why does the system wait before starting the heating again?", "t10", "5 minutes"),
    ("c11", "How long is the warranty?", "t12", "3 year"),
    ("c12", "Within how many days can I return it?", "t13", "45 days"),
    ("c13", "How do I get a new password?", "t15", "Forgot password"),
    ("c14", "How many people can share the home?", "t16", "8 people"),
    ("c15", "How long is the temperature history kept?", "t17", "12 months"),
    ("c16", "When do firmware updates install?", "t24", "02:00"),
    ("s01", "How many Thermio One units were sold in total?", "sales", exp["s01"]),
    ("s02", "Which region sold the most Thermio Line units in March?", "sales", exp["s02"]),
    ("s03", "What was the total revenue in June?", "sales", exp["s03"]),
    ("s04", "How many Thermio Sensor units were sold in the North region?", "sales", exp["s04"]),
    ("u01", "Does the thermostat work with voice assistants?", None, None),
    ("u02", "What is the price of a Thermio One?", None, None),
    ("u03", "Who founded the company?", None, None),
    ("u04", "How do I fix error E99?", None, None),
]
# dev set: used ONLY to choose the retrieval weight and the refusal threshold
DEV = [("d1", "Which wifi bands are supported?", "t01"), ("d2", "How cold can eco mode keep the house in winter?", "t06"),
       ("d3", "What happens if my account is deleted?", "t17"), ("d4", "Can I get my money back?", "t13"),
       ("d5", "How do sensors in other rooms work?", "t21"), ("d6", "Is my voice recorded?", "t25"),
       ("d7", "What if the screen stays dark?", "t08"), ("d8", "How big is the device?", "t22"),
       ("d9", "What does E45 mean?", "t10"), ("d10", "Do I need a C-wire?", "t02"),
       ("d11", "Is error E21 serious?", "t09"), ("d12", "firmware update time", "t24")]
OFF_TOPIC = ["What is the capital of France?", "How do I bake sourdough bread?", "Who won the football match yesterday?", "What is the speed of light?"]
print(len(BATTERY), "preguntas de la batería,", len(DEV), "preguntas de desarrollo")

referencias de las preguntas SQL: {'s01': '2465', 's02': 'South', 's03': '154176', 's04': '1251'}
24 preguntas de la batería, 12 preguntas de desarrollo


## 3. Recuperación: trocear por estructura y elegir el peso de la fusión midiendo

La base de conocimiento ya tiene estructura: un pasaje por tema, cada uno con un título de sección, así que el
fragmento es el pasaje y el título forma parte del texto buscable (módulo 01). La recuperación puede combinar BM25
y embeddings con RRF (módulo 02). En el módulo 02 los pesos iguales no eran los mejores; aquí **medimos** el peso
de BM25 sobre el conjunto de desarrollo en lugar de suponerlo, y aceptamos lo que diga.

In [3]:
STOP = set("a an and are as at be by can do does for from how i if in is it of on or that the this to what when which who with you my me".split())
tokens = lambda text: [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")

class Index:
    # chunks + BM25 + embeddings over a list of (id, section, text) passages
    def __init__(self, passages):
        self.ids = [p[0] for p in passages]
        self.texts = [f"{p[1]}. {p[2]}" for p in passages]
        self.tf = [defaultdict(int) for _ in self.texts]
        for i, t in enumerate(self.texts):
            for w in tokens(t):
                self.tf[i][w] += 1
        self.len = np.array([sum(d.values()) for d in self.tf], float)
        df = defaultdict(int)
        for d in self.tf:
            for w in d:
                df[w] += 1
        n = len(self.texts)
        self.idf = {w: math.log(1 + (n - c + 0.5) / (c + 0.5)) for w, c in df.items()}
        self.emb = embedder.encode(self.texts, normalize_embeddings=True)
    def bm25(self, q):
        s = np.zeros(len(self.texts))
        for w in set(tokens(q)):
            if w in self.idf:
                for i, d in enumerate(self.tf):
                    f = d.get(w, 0)
                    if f:
                        s[i] += self.idf[w] * f * 2.5 / (f + 1.5 * (0.25 + 0.75 * self.len[i] / self.len.mean()))
        return s
    def cosine(self, q):
        return self.emb @ embedder.encode(q, normalize_embeddings=True)
    def fused(self, q, w_bm25):
        tot = np.zeros(len(self.texts))
        for sc, w in ((self.bm25(q), w_bm25), (self.cosine(q), 1.0)):
            for rank, i in enumerate(np.argsort(-sc), 1):
                tot[i] += w / (60 + rank)
        return tot

index = Index(KB)

def mrr(idx, w):
    r = [list(np.argsort(-idx.fused(q, w))).index(idx.ids.index(g)) + 1 for _, q, g in DEV]
    return float(np.mean([1 / x for x in r])), float(np.mean([x == 1 for x in r]))

print("peso de BM25 (peso semántico = 1) frente al conjunto de DESARROLLO:")
dev_scores = {}
for w in (0, 0.25, 0.5, 1.0):
    dev_scores[w] = mrr(index, w)
    print(f"  peso {w:<5} MRR {dev_scores[w][0]:.3f}   hit@1 {dev_scores[w][1]:.2f}")
W_BM25 = max(dev_scores, key=lambda w: (dev_scores[w][0], -w))          # best MRR; ties go to the smaller (simpler) weight
print("peso de BM25 elegido:", W_BM25)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2735.79it/s]

peso de BM25 (peso semántico = 1) frente al conjunto de DESARROLLO:


  peso 0     MRR 0.958   hit@1 0.92


  peso 0.25  MRR 0.847   hit@1 0.75


  peso 0.5   MRR 0.833   hit@1 0.75


  peso 1.0   MRR 0.819   hit@1 0.75
peso de BM25 elegido: 0


En estas preguntas de desarrollo, añadir BM25 a la fusión **no ayudó**: el mejor peso es el que muestra la tabla, y
con el peso en cero el "híbrido" es solo semántico. Es un resultado legítimo de medir (en el módulo 02 las
preguntas con identificadores favorecían un peso pequeño de BM25; esta base de conocimiento es pequeña, sus
pasajes son distintivos y la mayoría de las preguntas de desarrollo son paráfrasis). Sería un error mantener un
peso solo porque "lo híbrido es la mejor práctica". Aun así, BM25 no se desperdicia: se queda en el sistema como el
**segundo ranker independiente** que usa el supervisor en la sección 7. Repite la selección del peso cada vez que
cambie el corpus.

## 4. Enrutador y SQL seguro

Algunas preguntas las responde un pasaje y otras un número. Un **enrutador** decide, y debe ser conservador: una
pregunta va a SQL solo si pide claramente una cantidad de la tabla de ventas. El camino SQL sigue el módulo 06: el
"modelo" (aquí un sustituto basado en reglas) escribe **un** `SELECT`, un **guardián** lo valida y DuckDB lo
ejecuta sobre una vista registrada.

In [4]:
MONTHS = {m: f"2026-{i:02d}" for i, m in enumerate(["january", "february", "march", "april", "may", "june"], 1)}

def route(question):
    q = question.lower()
    return "sql" if re.search(r"\b(sold|sales|revenue)\b", q) else "passage"

def text_to_sql(question):
    # stand-in for the model that writes the SELECT: a rule-based translation of the few shapes we support
    q = question.lower()
    where = []
    for p in PRICE:
        if p.lower() in q:
            where.append(f"product = '{p}'")
    for reg in ("north", "south"):
        if reg in q:
            where.append(f"region = '{reg.title()}'")
    for m, key in MONTHS.items():
        if m in q:
            where.append(f"month = '{key}'")
    metric = "revenue_eur" if "revenue" in q else "units"
    clause = (" WHERE " + " AND ".join(where)) if where else ""
    if re.search(r"which (region|product)", q):
        col = "region" if "region" in q else "product"
        return f"SELECT {col}, SUM({metric}) AS total FROM sales{clause} GROUP BY {col} ORDER BY total DESC LIMIT 1"
    return f"SELECT SUM({metric}) AS total FROM sales{clause}"

FORBIDDEN = re.compile(r"\b(insert|update|delete|drop|alter|create|attach|copy|pragma|install|load|read_\w+)\b", re.I)

def guard(sql):
    body = sql.strip().rstrip(";")
    if ";" in body or not re.match(r"(?is)^select\b", body):
        return False, "solo se permite un único SELECT"
    if FORBIDDEN.search(body):
        return False, "palabra clave prohibida"
    if set(re.findall(r"(?i)\b(?:from|join)\s+(\w+)", body)) - {"sales"}:
        return False, "tabla desconocida"
    return True, body

for bad in ["DROP TABLE sales", "SELECT * FROM sales; DELETE FROM sales", "SELECT * FROM read_csv('/etc/passwd')", "SELECT * FROM secrets"]:
    print(f"{bad[:44]:46s}-> {guard(bad)}")
print(text_to_sql("Which region sold the most Thermio Line units in March?"))

DROP TABLE sales                              -> (False, 'solo se permite un único SELECT')
SELECT * FROM sales; DELETE FROM sales        -> (False, 'solo se permite un único SELECT')
SELECT * FROM read_csv('/etc/passwd')         -> (False, 'palabra clave prohibida')
SELECT * FROM secrets                         -> (False, 'tabla desconocida')
SELECT region, SUM(units) AS total FROM sales WHERE product = 'Thermio Line' AND month = '2026-03' GROUP BY region ORDER BY total DESC LIMIT 1


## 5. Una respuesta tipada que no puede existir sin una cita

Cada respuesta sale del sistema como un objeto **Pydantic**. El propio tipo rechaza una respuesta que no sea un
rechazo y no tenga cita, de modo que un fallo del generador no puede producir en silencio una afirmación sin
respaldo. El generador es el sustituto determinista de los módulos anteriores: devuelve la frase del pasaje
principal más cercana a la pregunta. La regla de rechazo se calibra con el conjunto de desarrollo y con preguntas
fuera de tema, nunca con la batería.

In [5]:
class Citation(BaseModel):
    n: int
    source: str                      # a passage id, or "sales" for a database result

class Answer(BaseModel):
    question: str
    route: Literal["passage", "sql", "refuse"]
    text: str
    citations: list[Citation] = []
    @model_validator(mode="after")
    def needs_citation(self):
        if self.route != "refuse" and not self.citations:
            raise ValueError("an answer that is not a refusal must cite a source")
        return self

# ----- refusal threshold, from the dev set (answerable) and off-topic questions (not answerable) -----
dev_top = [index.cosine(q).max() for _, q, _ in DEV]
off_top = [index.cosine(q).max() for q in OFF_TOPIC]
T_COS = (max(off_top) + min(dev_top)) / 2 if max(off_top) < min(dev_top) else max(off_top) + 0.01
print(f"coseno máximo: preguntas de desarrollo {min(dev_top):.2f} a {max(dev_top):.2f}; fuera de tema {min(off_top):.2f} a {max(off_top):.2f}  ->  rechazar por debajo de {T_COS:.2f}")

REFUSAL = "No lo encuentro en la documentación de Thermio."

def best_sentence(question, text):
    sents = [s for s in re.split(r"(?<=[.!?])\s+", text) if s]
    sims = embedder.encode(sents, normalize_embeddings=True) @ embedder.encode(question, normalize_embeddings=True)
    return sents[int(np.argmax(sims))]

def answer_question(question, idx, w=None):
    # returns (Answer, details); details feed the trace and the supervisor
    w = W_BM25 if w is None else w
    if route(question) == "sql":
        ok, sql = guard(text_to_sql(question))
        if ok:
            row = con.execute(sql).fetchall()[0]
            value = row[0] if len(row) == 1 else row[0]
            return Answer(question=question, route="sql", text=f"{value} [1]", citations=[Citation(n=1, source="sales")]), dict(sql=sql, pieces=[], top_cos=1.0)
        return Answer(question=question, route="refuse", text=REFUSAL), dict(sql=sql, pieces=[], top_cos=0.0)
    cos = idx.cosine(question)
    order = list(np.argsort(-idx.fused(question, w)))[:3]
    if cos.max() < T_COS:
        return Answer(question=question, route="refuse", text=REFUSAL), dict(pieces=[idx.ids[i] for i in order], top_cos=float(cos.max()))
    top = order[0]
    sentence = best_sentence(question, idx.texts[top].split(". ", 1)[1])
    try:
        ans = Answer(question=question, route="passage", text=f"{sentence} [1]", citations=[Citation(n=1, source=idx.ids[top])])
    except ValidationError:
        ans = Answer(question=question, route="refuse", text=REFUSAL)
    return ans, dict(pieces=[idx.ids[i] for i in order], top_cos=float(cos.max()))

for q in ["Can I connect the thermostat to a 5 GHz network?", "Which region sold the most Thermio Line units in March?", "What is the capital of Peru?"]:
    a, _ = answer_question(q, index)
    print(f"{q}\n   [{a.route}] {a.text}   cites {[c.source for c in a.citations]}")

coseno máximo: preguntas de desarrollo 0.33 a 0.79; fuera de tema 0.07 a 0.17  ->  rechazar por debajo de 0.25
Can I connect the thermostat to a 5 GHz network?
   [passage] If the thermostat shows Offline, restart the router, then hold the ring button for 10 seconds to reconnect. [1]   cites ['t07']
Which region sold the most Thermio Line units in March?
   [sql] South [1]   cites ['sales']
What is the capital of Peru?
   [refuse] No lo encuentro en la documentación de Thermio.   cites []


## 6. Ejecutar la batería, con una traza por cada petición

Ahora el sistema responde las 24 preguntas de la batería. Cada petición escribe una **traza** (módulo 09): la ruta,
los pasajes que vio, el recuento de tokens, el coste (con precios supuestos) y una latencia simulada. La
calificación frente a las referencias se calcula aparte y es lo que intentamos mejorar.

In [6]:
PRICE_IN, PRICE_OUT = 0.035, 0.14                  # ASSUMED USD per million tokens for the generation model
TRACE_LOG = []

def grade(ans, gold, fact):
    # correct = contains the reference fact; for an unanswerable question (gold None) correct = a refusal
    if gold is None:
        return int(ans.route == "refuse")
    return int(ans.route != "refuse" and fact.lower() in ans.text.lower())

def run_battery(idx, label, battery=None):
    r = random.Random(5)
    rows = []
    for qid, q, gold, fact in (battery or BATTERY):
        ans, det = answer_question(q, idx)
        n_in = round((len(q.split()) + 60 * (ans.route == "passage")) * 1.3) or 1
        n_out = round(len(ans.text.split()) * 1.3)
        cost = (n_in * PRICE_IN + n_out * PRICE_OUT) / 1e6
        latency = (6 if ans.route != "sql" else 12) + (120 + 8 * n_out + r.random() * 30 if ans.route == "passage" else 0)
        TRACE_LOG.append(dict(run=label, id=qid, route=ans.route, tokens_in=n_in, tokens_out=n_out, cost=cost, latency=latency))
        rows.append(dict(id=qid, question=q, gold=gold, fact=fact, answer=ans, details=det, correct=grade(ans, gold, fact)))
    return rows

before = run_battery(index, "before")
c = np.array([r["correct"] for r in before])
groups = {"preguntas de pasaje (16)": [r for r in before if r["id"].startswith("c")], "preguntas SQL (4)": [r for r in before if r["id"].startswith("s")],
          "sin respuesta posible (4)": [r for r in before if r["id"].startswith("u")]}
print(f"correctas en total: {c.sum()} de {len(c)} ({c.mean():.0%})")
for name, g in groups.items():
    print(f"  {name:26s}{sum(x['correct'] for x in g)} de {len(g)}")
print("\nrespuestas incorrectas:")
for r in before:
    if not r["correct"]:
        print(f"  {r['id']} {r['question']}\n      -> [{r['answer'].route}] {r['answer'].text}   (referencia {r['fact']!r})")

correctas en total: 18 de 24 (75%)
  preguntas de pasaje (16)  13 de 16
  preguntas SQL (4)         4 de 4
  sin respuesta posible (4) 1 de 4

respuestas incorrectas:
  c01 Can I connect the thermostat to a 5 GHz network?
      -> [passage] If the thermostat shows Offline, restart the router, then hold the ring button for 10 seconds to reconnect. [1]   (referencia '2.4 GHz')
  c03 Does it work with electric baseboard heaters?
      -> [passage] Thermio One is not compatible with 120 V or 240 V line-voltage baseboard heaters. [1]   (referencia 'Thermio Line')
  c11 How long is the warranty?
      -> [passage] A faulty unit under warranty is replaced with a refurbished or new device and shipped within 3 business days of approval. [1]   (referencia '3 year')
  u01 Does the thermostat work with voice assistants?
      -> [passage] A remote room sensor (sold separately) measures temperature and occupancy up to 10 m from the thermostat. [1]   (referencia None)
  u02 What is the price of a Th

## 7. El supervisor

En producción, el supervisor es un segundo modelo que lee la pregunta, los pasajes y la respuesta y devuelve un
veredicto (módulo 08). Aquí es un **sustituto barato**, y tiene la única propiedad que también tiene el real:
**no conoce la respuesta de referencia**. Usa una señal clásica de incertidumbre que no necesita llamar a ningún
modelo: **¿coinciden dos rankers independientes?** La respuesta se construyó con el pasaje que más gustó al
recuperador. A un ranker BM25 puro, que funciona de otra manera que el recuperador, se le pregunta qué pasaje
citaría *él*. Si ambos discrepan, o BM25 no encuentra nada, la respuesta se marca con **baja confianza** y una
persona (o un modelo más potente) debería mirarla.

* un **rechazo** es *justificado* si nada del índice se parece a la pregunta, y una *respuesta omitida* si algo sí;
* una **respuesta de la base de datos** está *respaldada* (el guardián y la consulta son la evidencia);
* una **respuesta de pasaje** está *respaldada* si los rankers coinciden y es de *baja confianza* en caso contrario;
* de cada marca nombra los dos pasajes candidatos, que es donde está el arreglo.

In [7]:
def supervise(row, idx):
    ans, q = row["answer"], row["question"]
    near = idx.cosine(q)
    j = int(np.argmax(near))
    name = lambda k: f"{idx.ids[k]} ({idx.texts[k].split('. ')[0]})"
    if ans.route == "refuse":
        if near[j] >= T_COS + 0.1:                  # a passage clearly related exists: the refusal may be a miss
            return dict(verdict="omitida", missing=name(j))
        return dict(verdict="rechazo justificado", missing="")
    if ans.route == "sql":
        return dict(verdict="respaldada", missing="")
    b = idx.bm25(q)
    cited = idx.ids.index(ans.citations[0].source)
    if b.max() > 0 and int(np.argmax(b)) == cited:
        return dict(verdict="respaldada", missing="")
    return dict(verdict="baja confianza", missing=f"el recuperador eligió {name(cited)}; BM25 prefiere " + (name(int(np.argmax(b))) if b.max() > 0 else "nada (ninguna palabra coincide)"))

for r in before:
    r["judge"] = supervise(r, index)
verdicts = Counter(r["judge"]["verdict"] for r in before)
print(", ".join(f"{k}: {v}" for k, v in verdicts.most_common()), "\n")

# how does it behave on the DEV questions, whose answers we can check against the dev references?
dev_rows = [dict(question=q, answer=answer_question(q, index)[0]) for _, q, _ in DEV]
dev_flag = [supervise(r, index)["verdict"] == "baja confianza" for r in dev_rows]
dev_wrong = [(r["answer"].citations[0].source if r["answer"].citations else None) != g for r, (_, _, g) in zip(dev_rows, DEV)]
caught = sum(f and w for f, w in zip(dev_flag, dev_wrong))
print(f"en las {len(DEV)} preguntas de desarrollo marca {sum(dev_flag)}; {sum(dev_wrong)} respuestas eran realmente incorrectas; {caught} de las marcas eran errores reales, {sum(dev_wrong) - caught} respuestas incorrectas no se marcaron")

# does the supervisor agree with the references on the battery? (it never saw them)
flag_bad = np.array([r["judge"]["verdict"] in ("omitida", "baja confianza") for r in before])
truth_bad = np.array([not r["correct"] for r in before])
print(f"\n{'':28s}{'sin marcar':>14s}{'marcadas':>10s}")
for label, t in (("correctas (según la referencia)", False), ("incorrectas (según la referencia)", True)):
    print(f"{label:28s}{int(((truth_bad == t) & ~flag_bad).sum()):>14d}{int(((truth_bad == t) & flag_bad).sum()):>10d}")
print(f"\nmarcó {int((flag_bad & truth_bad).sum())} de {truth_bad.sum()} respuestas incorrectas (exhaustividad), {int((flag_bad & truth_bad).sum())} de {flag_bad.sum()} marcas eran errores reales (precisión)")

respaldada: 19, baja confianza: 4, rechazo justificado: 1 



en las 12 preguntas de desarrollo marca 5; 1 respuestas eran realmente incorrectas; 1 de las marcas eran errores reales, 0 respuestas incorrectas no se marcaron

                                sin marcar  marcadas
correctas (según la referencia)            18         0
incorrectas (según la referencia)             2         4

marcó 4 de 6 respuestas incorrectas (exhaustividad), 4 de 4 marcas eran errores reales (precisión)


Lee la tabla cruzada con honestidad. En la batería, el triaje marcó las respuestas incorrectas en las que los dos
rankers **discrepan**, y cada marca era un error real. Se le escaparon las incorrectas en las que ambos rankers
coincidían en el pasaje correcto y nuestro generador aun así escogió de él una frase incompleta o equivocada: el
juez mira *qué pasaje*, no *si la frase responde a la pregunta*. En las preguntas de desarrollo, parafraseadas y
cortas, **marca de más**, porque BM25 es débil cuando los usuarios no usan las palabras del documento (mira la
línea de desarrollo de arriba). Esas son las dos caras de un juez barato: se le escapan algunos errores reales y
da algunas falsas alarmas. Un juez LLM real, que lea los pasajes y la respuesta, cierra buena parte de esa
brecha, a un coste por respuesta juzgada que el presupuesto del módulo 10 permite calcular. Usa esta tabla para
decidir *cuánto fiarte del resumen nocturno*, no como prueba de que el sistema está bien.

## 8. El resumen diario

La parte del supervisor que escala es el **resumen**: no 24 veredictos que leer, sino una página que dice qué ha
cambiado y dónde mirar.

In [8]:
tl = [t for t in TRACE_LOG if t["run"] == "before"]
costs, lats = np.array([t["cost"] for t in tl]), np.array([t["latency"] for t in tl])
print(f"peticiones {len(tl)}   coste ${costs.sum():.6f} en total, ${costs.mean():.7f} por respuesta   latencia p50 {np.median(lats):.0f} ms, p95 {np.percentile(lats, 95):.0f} ms")
print("rutas: " + ", ".join(f"{k} {v}" for k, v in Counter(t["route"] for t in tl).most_common()))
print("veredictos: " + ", ".join(f"{k} {v}" for k, v in verdicts.most_common()))
print("\nrequieren atención (vista del supervisor, sin usar referencias):")
for r in before:
    if r["judge"]["verdict"] in ("omitida", "unsupported"):
        print(f"  {r['id']} [{r['judge']['verdict']}] {r['question']}\n        closest passage in the index: {r['judge']['missing']}")

peticiones 24   coste $0.000115 en total, $0.0000048 por respuesta   latencia p50 279 ms, p95 379 ms
rutas: passage 19, sql 4, refuse 1
veredictos: respaldada 19, baja confianza 4, rechazo justificado 1

requieren atención (vista del supervisor, sin usar referencias):


## 9. Análisis de fallos de recuperación y un arreglo de la documentación

Ahora, los fallos que las referencias dejaron al descubierto. Para cada pregunta de pasaje incorrecta nos hacemos
la única pregunta que decide el remedio: **¿en qué puesto del ranking estaba el pasaje que contiene la respuesta, y
qué palabras usó la pregunta que el pasaje no usa?** El puesto 1 con la frase equivocada es un problema de
generación; un puesto fuera de los 3 primeros es un fallo de recuperación, y la brecha de vocabulario indica cómo
cerrarla en la documentación.

In [9]:
def analyse(rows, idx):
    for r in rows:
        if r["correct"] or r["gold"] in (None, "sales"):
            continue
        order = [idx.ids[i] for i in np.argsort(-idx.fused(r["question"], W_BM25))]
        rank = order.index(r["gold"]) + 1
        gap = sorted(set(tokens(r["question"])) - set(tokens(idx.texts[idx.ids.index(r["gold"])])))
        print(f"{r['id']} {r['question']}\n    pasaje de referencia {r['gold']} en el puesto {rank} -> {'fallo de recuperación' if rank > 3 else 'entre los 3 primeros pero sin usar'}; "
              f"palabras de la pregunta que no están en el pasaje: {gap}")

analyse(before, index)
traps = [r for r in before if r["gold"] is None and not r["correct"]]
for r in traps:
    print(f"\n{r['id']} {r['question']}\n    respondió desde {r['answer'].citations[0].source if r['answer'].citations else '-'}: {r['answer'].text}   (nada en la documentación responde a esto)")

c01 Can I connect the thermostat to a 5 GHz network?
    pasaje de referencia t01 en el puesto 2 -> entre los 3 primeros pero sin usar; palabras de la pregunta que no están en el pasaje: ['connect', 'thermostat']
c03 Does it work with electric baseboard heaters?
    pasaje de referencia t03 en el puesto 1 -> entre los 3 primeros pero sin usar; palabras de la pregunta que no están en el pasaje: ['electric', 'work']
c11 How long is the warranty?
    pasaje de referencia t12 en el puesto 2 -> entre los 3 primeros pero sin usar; palabras de la pregunta que no están en el pasaje: ['long']

u01 Does the thermostat work with voice assistants?
    respondió desde t21: A remote room sensor (sold separately) measures temperature and occupancy up to 10 m from the thermostat. [1]   (nada en la documentación responde a esto)

u02 What is the price of a Thermio One?
    respondió desde t12: Thermio One has a 3 year limited warranty from the date of purchase. [1]   (nada en la documentación responde 

In [10]:
# THE FIX, written by reading the analysis above. Everything is a change to the DOCUMENTATION, the source of truth:
#  1) answer the question in the words users use (the answer sentence must contain the fact, in one place)
#  2) the questions the docs never covered get a passage that says so, or says where to go
FIXES = {
    "t01": " It cannot connect to 5 GHz networks, only to 2.4 GHz networks.",
    "t03": " Electric baseboard heaters are supported by Thermio Line, not by Thermio One.",
    "t12": " The warranty lasts 3 years.",
}
NEW = [
    ("t26", "Troubleshooting / Other error codes", "Only the error codes E21 and E45 are documented. For any other error code, contact support and include the code shown on the display."),
    ("t27", "Features / Voice assistants", "Thermio does not currently support voice assistants or third-party smart-home platforms."),
    ("t28", "Support / Pricing", "The price of a Thermio One is not listed in this documentation. See the store page for the current price."),
]
KB_FIXED = [(i, s, t + FIXES.get(i, "")) for i, s, t in KB] + NEW
for i in FIXES:
    print(f"{i}: añadido {FIXES[i]!r}")
for n in NEW:
    print(f"{n[0]}: nuevo pasaje '{n[1]}'")

t01: añadido ' It cannot connect to 5 GHz networks, only to 2.4 GHz networks.'
t03: añadido ' Electric baseboard heaters are supported by Thermio Line, not by Thermio One.'
t12: añadido ' The warranty lasts 3 years.'
t26: nuevo pasaje 'Troubleshooting / Other error codes'
t27: nuevo pasaje 'Features / Voice assistants'
t28: nuevo pasaje 'Support / Pricing'


Un arreglo en la documentación es un arreglo **para cada usuario futuro y para cada modelo futuro**, y por eso el
módulo 08 encamina las correcciones allí en lugar de parchear prompts. Fíjate en lo que *no* hicimos: no cambiamos
el código, los umbrales ni el peso de recuperación. El único cambio es la base de conocimiento. Ahora reindexamos y
volvemos a medir sobre la **misma batería**.

Una salvedad honesta, dicha antes de los números para que no pueda esconderse en ellos: tres de las cuatro
preguntas "sin respuesta posible" (`u01`, `u02`, `u04`) eran en realidad **huecos de documentación**, y el arreglo
da un pasaje a cada una. Para esas tres, el comportamiento *correcto* ya no es un rechazo, así que sus referencias
cambian (ahora esperan el dato del nuevo pasaje). Mover una referencia es legítimo cuando cambia el alcance de la
documentación, y peligroso si es lo único que se movió, así que informamos de **dos** resultados: las 24 preguntas
con las referencias actualizadas y las **20 preguntas cuya referencia no cambió**.

In [11]:
UPDATED = {"u01": ("t27", "does not currently support"), "u02": ("t28", "store page"), "u04": ("t26", "contact support")}
BATTERY_AFTER = [(i, q, *UPDATED[i]) if i in UPDATED else (i, q, g, f) for i, q, g, f in BATTERY]
index2 = Index(KB_FIXED)
after = run_battery(index2, "after", BATTERY_AFTER)

b_same = np.array([r["correct"] for r in before])                                           # old answers, old references
b_new = np.array([grade(r["answer"], g, f) for r, (_, _, g, f) in zip(before, BATTERY_AFTER)])   # old answers, UPDATED references
a_ok = np.array([r["correct"] for r in after])
unchanged = np.array([i not in UPDATED for i, *_ in BATTERY])

rng = np.random.default_rng(0)
def boot(b, a, mask):
    b, a = b[mask], a[mask]
    ix = rng.integers(0, len(a), (5000, len(a)))
    ci = lambda x: f"[{np.percentile(x, 2.5):.2f}, {np.percentile(x, 97.5):.2f}]"
    return b.mean(), ci(b[ix].mean(1)), a.mean(), ci(a[ix].mean(1)), a.mean() - b.mean(), ci(a[ix].mean(1) - b[ix].mean(1))

print(f"{'':42s}{'antes':>8s}{'IC 95%':>16s}{'después':>8s}{'IC 95%':>16s}{'cambio':>9s}{'IC 95%':>16s}")
for label, b, mask in (("20 preguntas, referencias sin cambios", b_same, unchanged), ("24 preguntas, referencias actualizadas", b_new, np.ones(24, bool))):
    r = boot(b, a_ok, mask)
    print(f"{label:42s}{r[0]:>8.2f}{r[1]:>16s}{r[2]:>8.2f}{r[3]:>16s}{r[4]:>+9.2f}{r[5]:>16s}")
fixed = [r["id"] for r, bb, aa in zip(after, b_new, a_ok) if aa and not bb]
broke = [r["id"] for r, bb, aa in zip(after, b_new, a_ok) if bb and not aa]
print("\nahora correctas:", fixed, "  ahora incorrectas (regresiones):", broke)
for r in after:
    if not r["correct"]:
        print(f"  siguen mal: {r['id']} {r['question']} -> {r['answer'].text}")

                                             antes          IC 95% después          IC 95%   cambio          IC 95%
20 preguntas, referencias sin cambios         0.86    [0.71, 1.00]    1.00    [1.00, 1.00]    +0.14    [0.00, 0.29]
24 preguntas, referencias actualizadas        0.75    [0.58, 0.92]    0.96    [0.88, 1.00]    +0.21    [0.04, 0.38]

ahora correctas: ['c01', 'c03', 'c11', 'u01', 'u04']   ahora incorrectas (regresiones): []
  siguen mal: u02 What is the price of a Thermio One? -> The price of a Thermio One is not listed in this documentation. [1]


Dos cosas que dicen los números y una que no pueden decir. Dicen **qué preguntas reparó el arreglo** (`c01`, `c03`,
`c11`, `u01`, `u04`) y, igual de importante, que **no rompió nada**: una comprobación de regresiones es la mitad del
valor de una batería fija. Dicen **cuán ancha es la incertidumbre**: en las 20 preguntas cuya referencia no
cambió, el intervalo de la ganancia va de 0,00 a 0,29, así que incluso una mejora clara apenas se distingue de no
haber cambiado nada, y una puntuación "después" de 1,00 da un intervalo degenerado, señal de que la batería ya es
demasiado fácil para distinguir sistemas mejores y necesita preguntas nuevas y más difíciles.

El único fallo que queda, `u02`, es un artefacto de un evaluador estricto: la respuesta dice que el precio no
figura en la documentación, lo cual es correcto, pero la referencia esperaba las palabras "store page". Las
referencias de texto son un instrumento romo (por eso el módulo 07 califica también con un juez), y el informe
honesto lo mantiene como fallo en lugar de relajar el evaluador después de ver el resultado.

**Los números no pueden decir que el sistema sea bueno**, solo que este cambio no lo empeoró en estas preguntas. El
tráfico real es más amplio que cualquier batería, y para eso está el resumen nocturno del supervisor (sección 8):
convierte la producción en la siguiente batería.

## 10. Antes de poner un sistema RAG delante de los usuarios

Una lista de comprobación, con cada punto señalando el módulo donde lo construiste:

- [ ] **Corpus:** cada documento tiene un responsable y una fecha; los fragmentos siguen la estructura del documento (01)
- [ ] **Recuperación:** el peso de la fusión elegido con un conjunto de desarrollo que no es tu conjunto de prueba, y reelegido cuando cambia el corpus (02)
- [ ] **Interfaces:** las llamadas a modelos pasan por una única capa de enrutado; cada entrada y salida tiene un esquema tipado (04)
- [ ] **Flujo:** los pasos son explícitos, con un enrutador y un camino de rechazo; ningún paso puede entrar en bucle infinito (05)
- [ ] **Números:** las cantidades salen de una consulta de solo lectura y con guardián, nunca de la memoria de un modelo (06)
- [ ] **Calidad:** una batería de preguntas con referencias tomadas de los datos, con trampas y preguntas sin respuesta posible, ejecutada en cada cambio (07)
- [ ] **Supervisión:** cada respuesta se juzga, los fallos se agrupan por causa y los arreglos van a la documentación (08)
- [ ] **Observabilidad:** una traza por petición con tokens, coste, latencia y puntuaciones; percentiles en un panel (09)
- [ ] **Presupuesto:** coste por respuesta estimado a partir de tus propias trazas; límites de uso y un tope de concurrencia (10)
- [ ] **Seguridad:** el texto recuperado se trata como no confiable; la salida se valida en código; los secretos fuera de las URL; los datos personales ocultos (10)
- [ ] **Honestidad:** el sistema dice "no lo encuentro" en lugar de adivinar, y el equipo conoce su tasa de rechazo
- [ ] **Responsabilidad:** alguien lee el resumen nocturno, y arreglar la documentación forma parte de su trabajo

## Resumen: del módulo al componente del proyecto final

| Módulo | Qué construiste | En este proyecto final |
|---|---|---|
| 01 | troceado, BM25, hit@k, MRR, rechazo | `Index.bm25`, fragmentos por sección, MRR del conjunto de desarrollo |
| 02 | embeddings, RRF, pesos | `Index.cosine`, `Index.fused`, el peso `W_BM25` elegido con el conjunto de desarrollo (BM25 pasó a ser la segunda opinión del supervisor) |
| 03 | LangChain y LlamaIndex | la misma recuperación escrita a mano: lo que abstraen los frameworks |
| 04 | LiteLLM y Pydantic | la `Answer` tipada y su validador; un único lugar para cambiar el generador |
| 05 | LangGraph | `route` y los pasos explícitos de `answer_question` |
| 06 | text-to-SQL seguro | `text_to_sql`, `guard`, DuckDB |
| 07 | evaluación | `BATTERY`, `DEV`, calificación frente a referencias independientes, IC por bootstrap |
| 08 | el supervisor | `supervise`, el resumen diario, el arreglo de la documentación y la reindexación |
| 09 | observabilidad | `TRACE_LOG` con tokens, coste y latencia |
| 10 | producción y seguridad | umbral de rechazo, modelo de coste, validación y los hábitos de la lista de comprobación |

## Ejercicios

1. Sustituye `best_sentence` por tomar como respuesta el pasaje principal completo. ¿Qué preguntas cambian, y qué
   dice eso sobre la frontera entre recuperación y generación?
2. Pon `W_BM25 = 1.0` después de la sección 3 y vuelve a ejecutar desde la sección 6. ¿Sigue el supervisor marcando
   los fallos? ¿Se debilita su segunda opinión cuando BM25 también guía la recuperación?
3. Añade una quinta trampa sin respuesta posible que comparta palabras con un pasaje real ("What is the warranty on
   the Thermio Line?"). ¿Qué capa debería detectarla: la recuperación, el generador, el supervisor o la documentación?
4. Escribe un arreglo de documentación nuevo para un fallo que tu supervisor marcó pero que este notebook no
   arregló, aplícalo e informa del antes y el después con el intervalo. ¿Cambió el intervalo tu conclusión?